In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random
import pickle
import math
import os
import tiktoken

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

# ── Hyper-parameters ─────────────────────────────────────────────────────────
TRAIN_FILE    = 'openwebtext_10gb.txt'
VAL_SPLIT     = 0.05          # last 5% of file bytes held out for validation
MODEL_OUT     = 'model-owt-2.pkl'
CHECKPOINT    = 'checkpoint.pt'

batch_size    = 32
block_size    = 128          # context length
max_iters     = 7000
learning_rate = 3e-4
eval_iters    = 100
eval_interval = 50
save_interval = 1000
n_embd        = 384
n_head        = 6
n_layer       = 6
dropout       = 0.1
grad_clip     = 1.0

print(device)

cuda


In [9]:
# Use tiktoken GPT-2 encoding
enc = tiktoken.get_encoding('gpt2')
vocab_size = enc.n_vocab
print(f'vocab_size: {vocab_size}')

vocab_size: 50257


In [10]:
# Map string encoder/decoder using tiktoken
# encode_ordinary skips special tokens (safer for raw web text)
encode = lambda s: enc.encode_ordinary(s)
decode = lambda l: enc.decode(l)

In [11]:
# ── Streaming on-disk data loader ─────────────────────────────────────────────
# Instead of loading the 10 GB file into RAM, we split it into train / val
# regions by byte offset and seek() to random positions for each batch.
# RAM usage stays ~constant regardless of file size.

CHUNK_BYTES = block_size * 6   # ~6 bytes/token average; over-read to be safe

file_size = os.path.getsize(TRAIN_FILE)
val_start = int(file_size * (1 - VAL_SPLIT))   # byte offset where val region begins

print(f'File size  : {file_size / 1e9:.2f} GB')
print(f'Train bytes: {val_start / 1e9:.2f} GB  |  Val bytes: {(file_size - val_start) / 1e6:.1f} MB')


def read_chunk(file_obj, byte_start: int, region_end: int) -> list:
    """Seek to byte_start, read a window, decode UTF-8, return token list."""
    read_size = min(CHUNK_BYTES * 4, region_end - byte_start)
    file_obj.seek(byte_start)
    raw = file_obj.read(read_size)
    raw = raw.decode('utf-8', errors='ignore')   # tolerate mid-codepoint start
    return encode(raw)


_file_handle = open(TRAIN_FILE, 'rb')   # single shared handle; stays open during training


def get_batch(split):
    if split == 'train':
        region_start, region_end = 0, val_start
    else:
        region_start, region_end = val_start, file_size

    x_list, y_list = [], []

    while len(x_list) < batch_size:
        max_start = region_end - CHUNK_BYTES * 4
        byte_start = random.randint(max(region_start, 0),
                                    max(region_start, max_start))

        tokens = read_chunk(_file_handle, byte_start, region_end)

        if len(tokens) < block_size + 1:
            continue   # rare near end of file — just retry

        tok_start = random.randint(0, len(tokens) - block_size - 1)
        x_list.append(tokens[tok_start : tok_start + block_size])
        y_list.append(tokens[tok_start + 1 : tok_start + block_size + 1])

    x = torch.tensor(x_list, dtype=torch.long, device=device)
    y = torch.tensor(y_list, dtype=torch.long, device=device)
    return x, y


# Quick sanity check
xb, yb = get_batch('train')
print(f'Batch shapes — x: {xb.shape}, y: {yb.shape}')

File size  : 10.83 GB
Train bytes: 10.29 GB  |  Val bytes: 541.3 MB
Batch shapes — x: torch.Size([32, 128]), y: torch.Size([32, 128])


In [12]:

class RMSNorm(nn.Module):
    """Root Mean Square Normalization (standard in LLaMA/Gemma)"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight



class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with fused QKV projection"""
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        
        # Fused QKV projection (more efficient than 3 separate projections)
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        # Causal mask
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))
                             .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        
        # Fused QKV computation
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        
        # Reshape for multi-head attention: (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # Use PyTorch's scaled_dot_product_attention (uses Flash Attention when available)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout if self.training else 0.0)
        
        # Reshape back: (B, n_head, T, head_dim) -> (B, T, C)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.c_proj(out)
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    """FFN with GELU activation (current standard)"""
    
    def __init__(self, n_embd):
        super().__init__()
        # 4x expansion is standard, GELU is the modern default
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block with Pre-RMSNorm (more stable training)"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = RMSNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        # Pre-LN: normalize BEFORE attention/FFN (GPT-2 style, more stable)
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
        # Weight tying: share weights between embedding and output projection
        # This is a key optimization that reduces params and improves performance
        self.token_embedding_table.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        
        # Scale residual projections (GPT-2 style initialization)
        for name, p in self.named_parameters():
            if name.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):
        """Generation with temperature and top-k sampling"""
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, _ = self.forward(index_cond)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling (better quality than pure sampling)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index


# ── Load checkpoint if available, otherwise init fresh model ──────────────────
start_iter    = 0
best_val_loss = float('inf')

model = GPTLanguageModel(vocab_size)
m = model.to(device)

if os.path.exists(CHECKPOINT):
    print(f'Resuming from {CHECKPOINT}...')
    ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=True)
    m.load_state_dict(ckpt['model'])
    start_iter    = ckpt['iter'] + 1
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resumed at iter {start_iter}, best val loss: {best_val_loss:.4f}')
else:
    print(f'Model parameters: {sum(p.numel() for p in m.parameters()):,}')

Resuming from checkpoint.pt...
  Resumed at iter 7001, best val loss: 5.1420


In [13]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [14]:
# Optimizer with cosine LR schedule
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)

# Reload optimizer state if resuming
if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=True)
    optimizer.load_state_dict(ckpt['optimizer'])

def get_lr(it):
    warmup_iters = 200
    min_lr = learning_rate / 10
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    decay_ratio = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

loss = None
for iter in range(start_iter, max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f'step {iter:4d} | train: {losses["train"]:.4f} | val: {losses["val"]:.4f}')
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']

    # Save checkpoint periodically
    if iter > 0 and iter % save_interval == 0:
        ckpt = {
            'iter'          : iter,
            'model'         : model.state_dict(),
            'optimizer'     : optimizer.state_dict(),
            'best_val_loss' : best_val_loss,
        }
        torch.save(ckpt, CHECKPOINT)
        print(f'  Checkpoint saved at iter {iter}')

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

if loss is not None:
    print(f'\nFinal loss: {loss.item():.4f}')
else:
    print(f'\nNo training steps were run (start_iter {start_iter} >= max_iters {max_iters}).')
with open(MODEL_OUT, 'wb') as f:
    pickle.dump(model, f)
print('Model saved!')
_file_handle.close()


No training steps were run (start_iter 7001 >= max_iters 7000).
Model saved!


In [15]:
prompt = 'What is the capital of America?'
context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context.unsqueeze(0), max_new_tokens=200, temperature=0.8, top_k=40)[0].tolist())
print(generated_chars)

What is the capital of America?

The “for-gash” of the anti-Semitism and “bam” with “the “exg” for “in” it’s the same thing, and it is the ‘em’ of the Islamic Union.

In the same, it means the government “re’s ” and “b” is not a very good thing to talk with the Trump campaign in a group of people’s words, or “the government of the United States,” in which you don’t think it is the “unco-h” that the law is not even the “stast of the United States.” But it’s clear why the media will look forward to the government in this situation, the truth is so much of our right to the end. It’s your words, �
